In [116]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import calendar
import missingno as msno
import seaborn as sn
%matplotlib inline

In [117]:
train=pd.read_csv('train.csv')
test=pd.read_csv('test.csv')

In [ ]:
print(train.head(5))
print(test.head(5))

In [ ]:
train.info()

In [ ]:
train.shape

In [ ]:
train.describe()

In [ ]:
train.dtypes

In [123]:

train['date']=train.datetime.apply(lambda x:x.split()[0])
train['hour']=train.datetime.apply(lambda x:x.split()[1].split(':')[0])
train['weekday']=train.date.apply(lambda dateString:calendar.day_name[datetime.strptime(dateString,"%Y-%m-%d").weekday()])
train['month']=train.date.apply(lambda dateString:calendar.month_name[datetime.strptime(dateString,"%Y-%m-%d").month])

In [ ]:
train.month.value_counts()

season - 1 = spring, 2 = summer, 3 = fall, 4 = winter

In [125]:
train['season']=train['season'].map({1:'Spring',2:'Summer',3:'Fall',4:'Winter'})

1: Clear, Few clouds, Partly cloudy, Partly cloudy
2: Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist
3: Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds
4: Heavy Rain + Ice Pallets + Thunderstorm + Mist, Snow + Fog

In [126]:
train['weather']=train['weather'].map({1:'Clear, Few clouds, Partly cloudy, Partly cloudy',2:'Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist',3:'Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds',4:'Heavy Rain + Ice Pallets + Thunderstorm + Mist, Snow + Fog'})


In [ ]:
train.head()

In [128]:
categorical_features_list=['hour','weekday','month','season','weather','holiday','workingday']
for feature in categorical_features_list:
    train[feature]=train[feature].astype('category')

In [ ]:
train.info()

In [130]:
train=train.drop('datetime',axis=1)

In [ ]:
train.info()

In [ ]:
msno.matrix(train,figsize=(12,5))

In [ ]:
fig,axes=plt.subplots(nrows=2,ncols=2)
fig.set_size_inches(12,10)
sn.boxplot(data=train,y="count",orient="v",ax=axes[0][0])
sn.boxplot(data=train,y="count",x="season",orient="v",ax=axes[0][1])
sn.boxplot(data=train,y="count",x="hour",orient="v",ax=axes[1][0])
sn.boxplot(data=train,y="count",x="weekday",orient="v",ax=axes[1][1])
axes[0][0].set(ylabel='Count',title='BoxPlot on Count')
axes[0][1].set(ylabel='Count',xlabel='Season',title='Boxplot on season and count')

#### Removing Outliers
A value is considered an outlier if its absolute value is more than three times the standard deviation of the values 

In [135]:
train_seventyfifth=train['count'].quantile(0.75)
train_twentyfifth=train['count'].quantile(0.25)
iqr=train_seventyfifth-train_twentyfifth
upper=train_seventyfifth+(1.5*iqr)
lower=train_twentyfifth-(1.5*iqr)
train_outliers_removed=train[(train['count']>lower) & (train['count']<upper) ]


In [ ]:
print("Training data with outliers",train.shape)
print("Training data without outliers",train_outliers_removed.shape)

In [ ]:
train.info()

In [ ]:
corrMatt=train[['temp','atemp','humidity','windspeed','casual','registered']].corr()
mask=np.array(corrMatt)
mask[np.tril_indices_from(mask)]=False
fig,ax=plt.subplots()
fig.set_size_inches(12,10)
sn.heatmap(corrMatt,mask=mask,vmax=8,square=True,annot=True)